# 05. Прогон одной модели по всем режимам

Один ноутбук = одна модель. Он сам проходит все режимы подряд: zero-shot и few-shot, каждый с JSON-схемой и без (4 прогона), и в конце пишет в Telegram.
Чтобы запустить другую модель, нужно поменять одну строку `MODEL` в ячейке «Настройки». Две модели можно считать одновременно: открой копию ноутбука во второй вкладке Colab и поставь другую `MODEL`.

Как запускать (Colab, A100):
1. В папку `vkr1_data` на Google Drive положить пять файлов: `dns_cards_500.json`, `gold_pairs.csv`, `split_ids.json`, `fewshot.jsonl`, `heldout.jsonl`.
2. Для Llama и Gemma принять условия на странице модели и добавить секрет `HF_TOKEN`.
3. Для Telegram добавить секреты `TG_TOKEN` (токен бота от @BotFather) и `TG_CHAT` (твой chat id). Без них ноутбук просто молчит.
4. Сначала поставить `N = 5` (проверка), потом `N = None`.
5. «Среда выполнения → Выполнить все». Если оборвалось, запусти ещё раз: готовые карточки пропускаются.

`real_test` не трогаем до этапа 09.

In [ ]:
# Один раз за сессию. После установки перезапусти сеанс (Среда выполнения → Перезапустить сеанс).
# flash-linear-attention нужен Qwen3.5: без него она считает в 5–10 раз медленнее.
%pip -q install -U transformers accelerate bitsandbytes xgrammar flash-linear-attention

In [ ]:
!pip list | grep -i "xgrammar\|flash-linear"

## Настройки
Меняй здесь только `MODEL` (и `N`, `SPLITS` при необходимости).

In [ ]:
!!rm -rf /root/.cache/huggingface/hub/*

[]

In [ ]:
import json, re, time, random, os
import numpy as np
import pandas as pd
import requests

MODELS = {
    # "qwen35_9b":       "Qwen/Qwen3.5-9B",                          # Alibaba, 9B, Apache 2.0
    # "gemma4_e4b":      "google/gemma-4-E4B-it",                    # Google, 4B эффективных параметров, Apache 2.0
    # "llama31_8b":      "meta-llama/Llama-3.1-8B-Instruct",         # Meta, 8B, лицензия Llama, нужен доступ
    # "ministral3_8b":   "mistralai/Ministral-3-8B-Instruct-2512-BF16",   # Mistral AI, 8B, Apache 2.0
    # "phi4_mini":       "microsoft/Phi-4-mini-instruct",            # Microsoft, 3.8B, MIT
    # "gigachat31_10b":  "ai-sage/GigaChat3.1-10B-A1.8B-bf16",       # Сбер, MoE 10B (1.8B активных), MIT
    # "tlite21_8b":      "t-tech/T-lite-it-2.1",                     # Т-Банк, 8B, Apache 2.0
    "yandex5_lite_8b": "yandex/YandexGPT-5-Lite-8B-instruct",      # Яндекс, 8B, своя лицензия
}

MODEL = "yandex5_lite_8b"     # <-- ключ из MODELS; для другой модели поменяй только эту строку
SPLITS = ["pilot"]      # на каких наборах: "pilot", "real_val", "heldout" (real_test только на этапе 09)
N = None                   # сколько карточек взять; None = все
K_SHOT = 5              # сколько примеров в few-shot
CONFIGS = [("zero-shot", False), ("zero-shot", True), ("few-shot", False), ("few-shot", True)]   # (режим, JSON-схема)

DATA = "data"           # папка с входными файлами
OUT = "results"         # сюда пишем результаты
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]

try:                    # в Colab данные и результаты лежат на Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA, OUT = "/content/drive/MyDrive/vkr1_data", "/content/drive/MyDrive/vkr1_results"
except ImportError:
    pass
os.makedirs(OUT, exist_ok=True)

print("Токен загружен:", bool(TG_BOT_TOKEN))
print("Chat ID загружен:", TG_CHAT_ID)

def tg(text):
    print(text)
    try:
        requests.post(f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage", data={"chat_id": TG_CHAT_ID, "text": text}, timeout=10)
    except Exception as e:
        print("telegram не отправился:", type(e).__name__)      # без текста ошибки: в нём был бы токен

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Токен загружен: True
Chat ID загружен: 1941343180


## Данные
Карточка = описание и эталон по семи атрибутам (`None`, если в тексте значения нет).
Для реальных карточек эталон берётся из `gold_pairs.csv`, для `heldout` он записан в самом файле.

In [ ]:
cards = {c["url"]: c["description"] for c in json.load(open(f"{DATA}/dns_cards_500.json", encoding="utf-8"))}
pairs = pd.read_csv(f"{DATA}/gold_pairs.csv", dtype=str)
gold_real = {}
for url, g in pairs.groupby("url"):
    gold_real[url] = {a: None for a in ATTRS}          # если атрибута нет в таблице, эталон null
    for _, r in g.iterrows():
        if pd.notna(r["эталон"]):
            gold_real[url][r["атрибут"]] = r["эталон"]
split = json.load(open(f"{DATA}/split_ids.json"))
fewshot = [json.loads(l) for l in open(f"{DATA}/fewshot.jsonl", encoding="utf-8")]
shots = random.Random(0).sample(fewshot, K_SHOT)      # одни и те же примеры для всех моделей

def get_items(name):
    if name == "heldout":
        syn = [json.loads(l) for l in open(f"{DATA}/heldout.jsonl", encoding="utf-8")]
        items = [{"id": f"heldout{i}", "text": r["description"], "gold": r["gold"]} for i, r in enumerate(syn)]
    else:
        items = [{"id": u, "text": cards[u], "gold": gold_real[u]} for u in split[name]]
    return items[:N] if N else items

ITEMS = {s: get_items(s) for s in SPLITS}
print({s: len(v) for s, v in ITEMS.items()})

{'pilot': 100}


## Промпт и схема ответа
Промпт одинаковый для всех моделей: одно сообщение пользователя, без system-роли, потому что шаблоны моделей с ней работают по-разному.
В few-shot к инструкции добавляются примеры из `fewshot.jsonl`.

In [ ]:
INSTRUCTION = """Извлеки из описания монитора значения семи характеристик и верни JSON.

Правила:
- Бери только то, что прямо написано в тексте описания. Если характеристики в тексте нет, пиши null.
- Не угадывай и не используй свои знания о модели монитора.
- Все значения строки в таком формате: диагональ — число в дюймах ("27"); разрешение — "2560x1440"; частота — число в герцах ("165"); тип матрицы — "IPS", "VA", "TN" или "OLED"; яркость — число в кд/м² ("300"); контрастность — число ("3000", а 1 млн к 1 пиши "1000000"); изогнутость — "да" или "нет".
- Ответ только JSON с ключами: диагональ, разрешение, частота, тип матрицы, яркость, контрастность, изогнутость."""

def make_prompt(text, mode):
    p = INSTRUCTION
    if mode == "few-shot":
        for i, ex in enumerate(shots, 1):
            p += f"\n\nПример {i}.\nОписание: {ex['description']}\nОтвет: {json.dumps(ex['gold'], ensure_ascii=False)}"
    return p + f"\n\nОписание: {text}\nОтвет:"

SCHEMA_JSON = {"type": "object",
               "properties": {a: {"type": ["string", "null"]} for a in ATTRS},
               "required": ATTRS, "additionalProperties": False}

## Загрузка модели
4-битная загрузка (nf4), вычисления в fp16, размышление (thinking) выключено. Если модель выдаёт мусор или `nan`, смотри раздел «Если что-то не работает» в README.

In [ ]:
from huggingface_hub import login
login(token=HF_ROKEN)

In [ ]:
%pip -q install mistral-common

In [ ]:
# GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import xgrammar as xgr
from xgrammar.contrib.hf import LogitsProcessor


name = MODELS[MODEL]
if MODEL == "gigachat31_10b":       # в config.json стоит "routed_scaling_factor": 1 (int), а transformers 5 ждёт float
    from huggingface_hub import snapshot_download
    src = snapshot_download(name)
    fixed = "/content/gigachat_fixed"
    os.makedirs(fixed, exist_ok=True)
    for f in os.listdir(src):         # веса не копируем, только ссылки
        if f != "config.json" and not os.path.exists(f"{fixed}/{f}"):
            os.symlink(os.path.realpath(f"{src}/{f}"), f"{fixed}/{f}")
    cfg = json.load(open(f"{src}/config.json"))
    cfg["routed_scaling_factor"] = float(cfg["routed_scaling_factor"])
    json.dump(cfg, open(f"{fixed}/config.json", "w"), indent=2)
    name = fixed
kw = {"fix_mistral_regex": True} if MODEL == "ministral3_8b" else {}     # у Ministral иначе неверная токенизация
tok = AutoTokenizer.from_pretrained(name, **kw)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
try:
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
except ValueError:                      # мультимодальные модели иногда грузятся другим классом
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(name, quantization_config=bnb, device_map="auto")
model.eval()
print(name, "| память после загрузки, ГБ:", round(torch.cuda.memory_allocated() / 1e9, 1))

tok_info = xgr.TokenizerInfo.from_huggingface(tok, vocab_size=model.get_output_embeddings().out_features)
compiled = xgr.GrammarCompiler(tok_info).compile_json_schema(json.dumps(SCHEMA_JSON), any_whitespace=False)   # один раз

def generate(prompt, use_schema):
    """возвращает (ответ модели, секунд, сколько токенов сгенерировано)"""
    chat = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    processors = [LogitsProcessor(compiled)] if use_schema else []     # для каждого запроса новый
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=200, do_sample=False, logits_processor=processors, stop_strings=["}"], tokenizer=tok)
    sec = time.time() - t0
    new = out[0][x["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), sec, len(new)

def reset_peak():
    torch.cuda.reset_peak_memory_stats()      # пик памяти считаем отдельно для каждого режима

config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/192k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.57MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

yandex/YandexGPT-5-Lite-8B-instruct | память после загрузки, ГБ: 6.1


## Метрики
Пара «атрибут–значение» считается верной, если значение совпало с эталоном. Если эталон `null`, а модель назвала значение, это ложное значение.
Считаем два варианта: `строго` (текст ответа как есть) и `с нормализацией` (значение приводится правилами из `03_eda`: «27 дюймов» → «27»). Разница между ними показывает, сколько ошибок даёт запись единиц (гипотеза H5).

In [ ]:
def parse(raw):
    """словарь по семи атрибутам или None, если ответ не разобрался как JSON"""
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        d = json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        d = None
    if not isinstance(d, dict):
        return None
    return {a: (None if d.get(a) in (None, "", "null") else str(d[a]).strip()) for a in ATTRS}

def number(s):
    return re.search(r"\d+(?:[.,]\d+)?", s).group().replace(",", ".")

def resolution(s):
    w, h = re.findall(r"\d+", s)[:2]
    return f"{w}x{h}" if int(w) >= 640 else None

def contrast(s):
    s = re.sub(r"(?<=\d)\s(?=\d{3}\b)", "", s.lower())
    v = float(number(s)) * (10**6 if re.search(r"m|м", s) else 1)
    return str(int(v)) if v >= 100 else None

NORM = {"диагональ": lambda s: f"{float(number(s)):g}", "разрешение": resolution, "частота": number,
        "тип матрицы": lambda s: s.upper().replace("QD-", ""), "яркость": number, "контрастность": contrast,
        "изогнутость": lambda s: s.lower()}

def norm(attr, v):
    try:
        return NORM[attr](v)
    except Exception:
        return v

def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0

def score(rows, soft):
    cnt = {a: [0, 0, 0] for a in ATTRS}          # tp, fp, fn по атрибутам
    false_null = []                              # для пар с эталоном null: 1, если модель назвала значение
    exact = []                                   # все семь атрибутов верны
    for r in rows:
        pred = r["pred"] or {a: None for a in ATTRS}
        ok = True
        for a in ATTRS:
            g, p = r["gold"][a], pred[a]
            if soft and p is not None:
                p = norm(a, p)
            if g is None:
                false_null.append(p is not None)
                if p is not None:
                    cnt[a][1] += 1; ok = False
            elif p == g:
                cnt[a][0] += 1
            else:
                cnt[a][2] += 1; ok = False
                if p is not None:
                    cnt[a][1] += 1
        exact.append(ok)
    tp, fp, fn = (sum(c[i] for c in cnt.values()) for i in range(3))
    macro = np.mean([f1(*c) for c in cnt.values() if sum(c)])
    return {"micro_f1": f1(tp, fp, fn), "macro_f1": macro, "exact": np.mean(exact),
            "false_null": np.mean(false_null), "per_attr": {a: round(f1(*c), 3) for a, c in cnt.items()}}

def make_summary(rows, split, mode, schema):
    strict, soft = score(rows, False), score(rows, True)
    sec = [r["sec"] for r in rows]
    summary = {"split": split, "model": MODEL, "mode": mode, "schema": schema, "n": len(rows),
               "hf_id": MODELS[MODEL], "k_shot": K_SHOT if mode == "few-shot" else 0, "дата": time.strftime("%Y-%m-%d %H:%M"),
               "micro_f1_строго": round(strict["micro_f1"], 3), "micro_f1_норм": round(soft["micro_f1"], 3),
               "macro_f1_норм": round(soft["macro_f1"], 3), "exact_норм": round(soft["exact"], 3),
               "ложные_на_null": round(soft["false_null"], 3),
               "невалидных": round(np.mean([r["pred"] is None for r in rows]), 3),
               "сек_среднее": round(np.mean(sec), 2), "сек_p95": round(np.percentile(sec, 95), 2),
               "токенов_среднее": round(np.mean([r["tokens"] for r in rows]), 1)}
    try:
        import torch, transformers
        summary["память_пик_ГБ"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
        summary["transformers"] = transformers.__version__
        summary["gpu"] = torch.cuda.get_device_name(0)       # время и память сравнимы только на одной и той же карте
    except Exception:
        pass
    return summary

def save_summary(summary):
    """строка в общий файл (старая строка с тем же набором, моделью и режимом заменяется)"""
    sp = f"{OUT}/summary.csv"
    old = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
    if len(old):
        same = ((old["split"] == summary["split"]) & (old["model"] == summary["model"])
                & (old["mode"] == summary["mode"]) & (old["schema"] == summary["schema"]))
        old = old[~same]
    pd.concat([old, pd.DataFrame([summary])]).to_csv(sp, index=False)

## Прогон
Для каждого набора и каждого из четырёх режимов: карточки идут по одной (batch 1), чтобы честно мерить задержку. Каждая готовая карточка сразу дописывается в файл, поэтому после обрыва можно просто запустить ячейку ещё раз.

In [ ]:
from tqdm import tqdm
import transformers
transformers.logging.set_verbosity_error()      # убирает повторяющееся предупреждение про max_length на каждой карточке

t_all = time.time()
lines = []
try:
    for sp_name in SPLITS:
        items = ITEMS[sp_name]
        for mode, schema in CONFIGS:
            tag = f"{sp_name}__{MODEL}__{mode}__{'schema' if schema else 'free'}"
            path = f"{OUT}/{tag}.jsonl"
            try:
                rows = list({r["id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}.values())     # дубли по id убираем
            except FileNotFoundError:          # файла ещё нет
                rows = []
            have = {r["id"] for r in rows}
            print("\n==", tag)
            reset_peak()
            bar = tqdm(total=len(items), initial=len(have), desc=tag)      # полоска: сколько готово, сколько осталось
            for it in items:
                if it["id"] in have:
                    continue
                raw, sec, ntok = generate(make_prompt(it["text"], mode), schema)
                row = {"id": it["id"], "gold": it["gold"], "raw": raw, "pred": parse(raw), "sec": round(sec, 2), "tokens": ntok}
                rows.append(row)
                for attempt in range(5):       # Google Drive иногда на пару секунд теряет папку
                    try:
                        os.makedirs(OUT, exist_ok=True)
                        with open(path, "a", encoding="utf-8") as f:
                            f.write(json.dumps(row, ensure_ascii=False) + "\n")
                        break
                    except FileNotFoundError:
                        if attempt == 4:
                            raise
                        time.sleep(5)
                bar.update(1)
                # if len(rows) - len(have) <= 2:
                    # bar.write("  ответ: " + raw[:200])
            bar.close()
            s = make_summary(rows, sp_name, mode, schema)
            save_summary(s)
            # print("  готово:", s["micro_f1_норм"], "F1 | ложные", s["ложные_на_null"], "|", s["сек_среднее"], "с на карточку")
            tg(f"{tag} готово: {s["micro_f1_норм"]} F1 | ложные {s["ложные_на_null"]} | {s["сек_среднее"]} с на карточку")
            lines.append(f"{sp_name} {mode} {'схема' if schema else 'свободно'}: F1 {s['micro_f1_норм']}, ложные {s['ложные_на_null']}, {s['сек_среднее']} с")
    tg(f"{MODEL} готово за {(time.time() - t_all) / 60:.0f} мин\n")
except Exception as e:
    tg(f"{MODEL}: ОШИБКА {type(e).__name__}: {str(e)[:300]}\n" + "\n".join(lines) + "\nЗапусти ячейку ещё раз: готовые карточки пропустятся.")
    raise


== pilot__yandex5_lite_8b__zero-shot__free


pilot__yandex5_lite_8b__zero-shot__free: 100%|██████████| 100/100 [00:00<?, ?it/s]


pilot__yandex5_lite_8b__zero-shot__free готово: 0.884 F1 | ложные 0.164 | 4.98 с на карточку

== pilot__yandex5_lite_8b__zero-shot__schema


pilot__yandex5_lite_8b__zero-shot__schema: 100%|██████████| 100/100 [00:00<?, ?it/s]

pilot__yandex5_lite_8b__zero-shot__schema готово: 0.896 F1 | ложные 0.143 | 4.12 с на карточку


...

pilot__yandex5_lite_8b__few-shot__free: 100%|██████████| 100/100 [00:00<?, ?it/s]

pilot__yandex5_lite_8b__few-shot__free готово: 0.917 F1 | ложные 0.088 | 5.04 с на карточку



== pilot__yandex5_lite_8b__few-shot__schema


pilot__yandex5_lite_8b__few-shot__schema: 100%|██████████| 100/100 [05:50<00:00,  4.32s/it]


pilot__yandex5_lite_8b__few-shot__schema готово: 0.917 F1 | ложные 0.091 | 4.36 с на карточку
yandex5_lite_8b готово за 6 мин



In [ ]:

s = pd.read_csv(f"{OUT}/summary.csv")
cols = ["split", "mode", "schema", "n", "micro_f1_норм", "ложные_на_null", "невалидных", "сек_среднее", "память_пик_ГБ"]
s[(s["model"] == MODEL) & s["split"].isin(SPLITS)][[c for c in cols if c in s]]

,split,mode,schema,n,micro_f1_норм,ложные_на_null,невалидных,сек_среднее,память_пик_ГБ
35,pilot,zero-shot,False,100,0.884,0.164,0.0,4.98,6.1
36,pilot,zero-shot,True,100,0.896,0.143,0.0,4.12,6.1
37,pilot,few-shot,False,100,0.917,0.088,0.0,5.04,6.1
38,pilot,few-shot,True,100,0.917,0.091,0.0,4.36,6.6


## Как читать результат
- `micro_f1_норм` — главная метрика. Если `micro_f1_строго` (в `summary.csv`) заметно ниже, модель правильно находит значения, но пишет их не в нужном формате.
- `ложные_на_null` — как часто модель придумывает значение, которого в тексте нет. Чем ниже, тем лучше.
- `невалидных` — доля ответов, которые не разобрались как JSON. С JSON-схемой это почти всегда 0.
- `сек_среднее` и `память_пик_ГБ` нужны для таблицы сравнения: задержка и сколько памяти заняла модель. Сравнимы только между прогонами на одной и той же карте.